# Corrected CAM-LDS MITRE ATT&CK LLM Fine-Tuning Notebook

This version fixes model collapse and class imbalance issues.

## Key Improvements:

1. **Reduced oversampling** (120 → 40) to prevent duplication artifacts
2. **Stricter label filtering** (MIN_SAMPLES=20) for reliability
3. **Top-N label selection** to improve class separability
4. **Optimized prompts** to reduce token usage
5. **Windows compatibility** - auto-detect and use fp16 on Windows
6. **Better diagnostics** - per-class accuracy, confusion matrix, dominance warnings
7. **Extended training** (3 → 5 epochs) for better convergence


In [52]:
!nvidia-smi

Fri May 15 00:54:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 581.83                 Driver Version: 581.83         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3050 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   44C    P8              3W /   70W |      91MiB /   4096MiB |     38%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [53]:
# Install dependencies (uncomment if needed)
# %%capture
# %pip install -U unsloth trl datasets accelerate bitsandbytes transformers peft scikit-learn pandas tqdm requests matplotlib
# %pip install unsloth_zoo

In [54]:
import os, re, json, zipfile, hashlib, requests, random, platform, sys
from pathlib import Path
from collections import defaultdict, Counter

import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

print(f"Platform: {platform.system()} | Python: {sys.version.split()[0]}")

Platform: Windows | Python: 3.12.3


In [55]:
# ============================================================================
# CONFIGURATION: Dataset Balancing & Platform Compatibility
# ============================================================================

BASE = Path.cwd()
RAW_DIR = BASE / "data" / "raw"
PROCESSED_DIR = BASE / "data" / "processed"
OUTPUT_DIR = BASE / "outputs"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Platform detection
IS_WINDOWS = platform.system() == "Windows"
IS_COLAB = "google.colab" in sys.modules

# Dataset configuration - reduced oversampling to prevent artifacts
MAX_PER_TECHNIQUE = 100  # Reduced from 200
MIN_SAMPLES_FOR_EVAL = 20  # Increased from 5 for reliability
BALANCED_SAMPLES_PER_CLASS = 40  # Reduced from 120 to avoid duplication
TOP_N_LABELS = 10  # NEW: Train only on top N techniques for better separability
NUM_EPOCHS = 5  # Increased from 3
NUM_EPOCHS = 1  # Increased from 3

# Model selection based on platform
if IS_WINDOWS:
    # Windows: use fp16 instead of 4-bit quantization
    MODEL_NAME = "unsloth/Qwen2.5-1.5B-Instruct"
    USE_4BIT = False
    print("⚠ Windows detected: using fp16 instead of 4-bit quantization")
    print("  Recommendation: Colab/Linux preferred for QLoRA training")
else:
    MODEL_NAME = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"
    USE_4BIT = True
    print("✓ Linux/Colab: using 4-bit quantization")

print(f"Model: {MODEL_NAME}")
print(f"Colab: {IS_COLAB}")

⚠ Windows detected: using fp16 instead of 4-bit quantization
  Recommendation: Colab/Linux preferred for QLoRA training
Model: unsloth/Qwen2.5-1.5B-Instruct
Colab: False


In [56]:
record_url = "https://zenodo.org/api/records/18861762"
target_name = "manifestations_raw.zip"
zip_path = RAW_DIR / target_name

if not zip_path.exists():
    record = requests.get(record_url, timeout=60).json()
    file_info = None
    for f in record.get("files", []):
        if f.get("key") == target_name:
            file_info = f
            break
    if file_info is None:
        raise RuntimeError("manifestations_raw.zip not found in Zenodo record.")
    download_url = file_info["links"]["self"]
    with requests.get(download_url, stream=True, timeout=60) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        with open(zip_path, "wb") as f, tqdm(total=total, unit="B", unit_scale=True) as pbar:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))
else:
    print("Already downloaded:", zip_path)

Already downloaded: d:\projects\camlds_mitre_llm_project\data\raw\manifestations_raw.zip


In [57]:
TECHNIQUE_RE = re.compile(r"T\d{4}(?:[.-]\d{3})?")

def clean_text(x):
    if x is None:
        return ""
    return str(x).replace("\n", " ").replace("\r", " ").strip()

def compact_json(obj, max_chars=500):
    try:
        s = json.dumps(obj, ensure_ascii=False, sort_keys=True)
    except Exception:
        s = str(obj)
    return s[:max_chars]

def stable_hash(text):
    return hashlib.sha1(text.encode("utf-8", errors="ignore")).hexdigest()[:16]

def extract_technique_id(path):
    matches = TECHNIQUE_RE.findall(str(path))
    if not matches:
        return None
    technique = matches[-1]
    return technique.replace("-", ".")

def alert_to_text(event):
    """Extract compact alert text to reduce token usage."""
    alert = event.get("alert", {}) or {}
    http = event.get("http", {}) or {}
    dns = event.get("dns", {}) or {}
    tls = event.get("tls", {}) or {}

    parts = [
        f"sig:{clean_text(alert.get('signature')[:40])}",
        f"cat:{clean_text(alert.get('category')[:30])}",
        f"sev:{clean_text(alert.get('severity'))}",
        f"proto:{clean_text(event.get('proto'))}",
        f"app:{clean_text(event.get('app_proto'))}",
        f"http:{clean_text(http.get('hostname')[:30])}",
        f"url:{clean_text(http.get('url')[:40])}",
        f"dns:{clean_text(dns.get('rrname')[:30])}",
        f"tls:{clean_text(tls.get('sni')[:30])}",
    ]
    return " | ".join([p for p in parts if p and not p.endswith(":")])

In [58]:
extract_dir = RAW_DIR / "manifestations_raw"

if not (extract_dir / "techniques").exists():
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(RAW_DIR)

eve_files = []
for pattern in ["**/eve.json", "**/eve.jsonl", "**/*eve*.json", "**/*eve*.jsonl"]:
    eve_files.extend(extract_dir.glob(pattern))

eve_files = sorted(set([f for f in eve_files if f.is_file()]))
print(f"Found eve files: {len(eve_files)}")

Found eve files: 8034


In [59]:
def clean_text(x, max_chars=None):
    if x is None:
        return ""
    text = str(x).replace("\n", " ").replace("\r", " ").strip()
    return text[:max_chars] if max_chars is not None else text

def alert_to_text(event):
    """Extract compact alert text to reduce token usage."""
    alert = event.get("alert", {}) or {}
    http = event.get("http", {}) or {}
    dns = event.get("dns", {}) or {}
    tls = event.get("tls", {}) or {}

    parts = [
        f"sig:{clean_text(alert.get('signature'), max_chars=40)}",
        f"cat:{clean_text(alert.get('category'), max_chars=30)}",
        f"sev:{clean_text(alert.get('severity'))}",
        f"proto:{clean_text(event.get('proto'))}",
        f"app:{clean_text(event.get('app_proto'))}",
        f"http:{clean_text(http.get('hostname'), max_chars=30)}",
        f"url:{clean_text(http.get('url'), max_chars=40)}",
        f"dns:{clean_text(dns.get('rrname'), max_chars=30)}",
        f"tls:{clean_text(tls.get('sni'), max_chars=30)}",
    ]
    return " | ".join([p for p in parts if p and not p.endswith(":")])

rows = []
seen_keys = set()

for path in tqdm(eve_files):
    technique_id = extract_technique_id(path)
    if technique_id is None:
        continue

    rel = str(path.relative_to(extract_dir))
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                event = json.loads(line)
            except json.JSONDecodeError:
                continue

            if event.get("event_type") != "alert":
                continue

            alert = event.get("alert", {}) or {}
            text = alert_to_text(event)

            if not text or text.count(":") < 3:
                continue

            key = f"{rel}:{line_no}:{text}:{technique_id}"
            alert_hash = stable_hash(key)
            if alert_hash in seen_keys:
                continue

            seen_keys.add(alert_hash)
            rows.append({
                "alert_id": alert_hash,
                "source_file": rel,
                "source_line": line_no,
                "gold_technique_id": technique_id,
                "alert_signature": clean_text(alert.get("signature")),
                "text": text,
            })

df = pd.DataFrame(rows)
print(f"Raw rows: {len(df)}")
print(f"Raw labels: {df.gold_technique_id.nunique()}")
print(f"Deduplicated alerts: {len(seen_keys)}")

# Cap very large classes
parts = []
for label, g in df.groupby("gold_technique_id"):
    parts.append(g.sample(min(len(g), MAX_PER_TECHNIQUE), random_state=42))
df = pd.concat(parts).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"After cap rows: {len(df)}")
print(f"After cap labels: {df.gold_technique_id.nunique()}")
print(f"\nTop 20 techniques by frequency:")
print(df["gold_technique_id"].value_counts().head(20))

  0%|          | 0/8034 [00:00<?, ?it/s]

Raw rows: 26250
Raw labels: 66
Deduplicated alerts: 26250
After cap rows: 2245
After cap labels: 66

Top 20 techniques by frequency:
gold_technique_id
T1219.000    100
T1033.000    100
T1595.003    100
T1614.000    100
T1518.000    100
T1069.000    100
T1201.000    100
T1057.000    100
T1595.002    100
T1016.000    100
T1007.000    100
T1087.000    100
T1049.000    100
T1615.000    100
T1082.000    100
T1083.000    100
T1059.004     82
T1204.001     70
T1110.001     59
T1105.000     49
Name: count, dtype: int64


In [60]:
# ============================================================================
# LABEL FILTERING & STRATIFIED SPLIT
# ============================================================================

counts = df["gold_technique_id"].value_counts()
valid_labels = counts[counts >= MIN_SAMPLES_FOR_EVAL].index.tolist()
df_eval = df[df["gold_technique_id"].isin(valid_labels)].copy()

print(f"Labels after MIN_SAMPLES_FOR_EVAL={MIN_SAMPLES_FOR_EVAL}: {len(valid_labels)}")
print(f"Rows after filtering: {len(df_eval)}")
print(f"Dropped: {sorted(set(df.gold_technique_id) - set(valid_labels))}")

# NEW: Filter to top N labels for better class separability
if TOP_N_LABELS is not None and TOP_N_LABELS > 0:
    top_labels = counts.head(TOP_N_LABELS).index.tolist()
    df_eval = df_eval[df_eval["gold_technique_id"].isin(top_labels)].copy()
    print(f"\n✓ Filtered to TOP_N_LABELS={TOP_N_LABELS}:")
    print(f"  {top_labels}")
    print(f"  Rows: {len(df_eval)}")

# Stratified split
train_df, test_df = train_test_split(
    df_eval,
    test_size=0.25,
    random_state=42,
    stratify=df_eval["gold_technique_id"]
)

print("\n=== Dataset Split ===")
print(f"Train (before balancing): {len(train_df)}")
print(f"Test: {len(test_df)}")
print(f"Train labels: {train_df.gold_technique_id.nunique()}")
print(f"Test labels: {test_df.gold_technique_id.nunique()}")

Labels after MIN_SAMPLES_FOR_EVAL=20: 27
Rows after filtering: 2055
Dropped: ['T1003.008', 'T1036.005', 'T1053.000', 'T1053.003', 'T1056.001', 'T1056.004', 'T1059.000', 'T1059.006', 'T1070.004', 'T1071.001', 'T1078.003', 'T1080.000', 'T1087.001', 'T1095.000', 'T1098.004', 'T1115.000', 'T1136.001', 'T1176.000', 'T1190.000', 'T1204.002', 'T1205.001', 'T1210.000', 'T1213.000', 'T1218.000', 'T1485.000', 'T1486.000', 'T1489.000', 'T1525.000', 'T1543.002', 'T1546.000', 'T1547.000', 'T1548.003', 'T1550.001', 'T1556.003', 'T1564.001', 'T1565.001', 'T1566.004', 'T1574.000', 'T1610.000']

✓ Filtered to TOP_N_LABELS=10:
  ['T1219.000', 'T1033.000', 'T1595.003', 'T1614.000', 'T1518.000', 'T1069.000', 'T1201.000', 'T1057.000', 'T1595.002', 'T1016.000']
  Rows: 1000

=== Dataset Split ===
Train (before balancing): 750
Test: 250
Train labels: 10
Test labels: 10


In [61]:
# ============================================================================
# MODERATE BALANCING: Avoid Aggressive Duplication
# ============================================================================

balanced_parts = []

for label, g in train_df.groupby("gold_technique_id"):
    if len(g) >= BALANCED_SAMPLES_PER_CLASS:
        sampled = g.sample(BALANCED_SAMPLES_PER_CLASS, random_state=42)
    else:
        # Moderate oversampling: target = min(40, len(g)*2)
        target = min(BALANCED_SAMPLES_PER_CLASS, len(g) * 2)
        sampled = g.sample(target, replace=True, random_state=42)
    balanced_parts.append(sampled)

train_balanced_df = pd.concat(balanced_parts).sample(frac=1, random_state=42).reset_index(drop=True)

print("=== Training Set After Balancing ===")
print(f"Total samples: {len(train_balanced_df)}")
print(f"\nClass distribution:")
print(train_balanced_df["gold_technique_id"].value_counts())
print(f"\nDistribution stats:")
print(train_balanced_df["gold_technique_id"].value_counts().describe())

=== Training Set After Balancing ===
Total samples: 400

Class distribution:
gold_technique_id
T1219.000    40
T1595.002    40
T1016.000    40
T1057.000    40
T1595.003    40
T1518.000    40
T1069.000    40
T1614.000    40
T1033.000    40
T1201.000    40
Name: count, dtype: int64

Distribution stats:
count    10.0
mean     40.0
std       0.0
min      40.0
25%      40.0
50%      40.0
75%      40.0
max      40.0
Name: count, dtype: float64


In [62]:
# ============================================================================
# PROMPT OPTIMIZATION: Shorter, More Efficient Prompts
# ============================================================================

def make_prompt(alert_text, allowed_labels):
    """Compact prompt to reduce token usage."""
    return (
        "Map this Suricata alert to exactly one MITRE ATT&CK technique ID.\n"
        "Return ONLY the technique ID (T1234 or T1234.001). No explanation.\n\n"
        f"Alert: {alert_text}"
    )

def make_training_text(row, tokenizer, allowed_labels):
    messages = [
        {"role": "system", "content": "You are a cybersecurity analyst mapping alerts to MITRE ATT&CK techniques."},
        {"role": "user", "content": make_prompt(row["text"], allowed_labels)},
        {"role": "assistant", "content": row["gold_technique_id"]},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

In [65]:
# ============================================================================
# MODEL LOADING: Windows-Compatible Setup
# ============================================================================

from unsloth import FastLanguageModel
from unsloth import is_bfloat16_supported
import warnings
import torch

warnings.filterwarnings("ignore")

# Force CUDA initialization early (helps with Windows DLL issues)
torch.cuda.is_available()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

max_seq_length = 1536

# Load model with platform-specific settings
if IS_WINDOWS:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=max_seq_length,
        load_in_4bit=False,
        dtype=torch.float16,
    )
    print("✓ Loaded with fp16 (Windows)")
else:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=max_seq_length,
        dtype=None,
        load_in_4bit=True,
    )
    print("✓ Loaded with 4-bit quantization (Linux/Colab)")

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print("✓ LoRA adapter configured")

OSError: [WinError 1114] A dynamic link library (DLL) initialization routine failed. Error loading "c:\Users\Aryan Gupta\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.

In [ ]:
# ============================================================================
# DATASET PREPARATION & DIAGNOSTICS
# ============================================================================

from datasets import Dataset

allowed_labels = sorted(train_df["gold_technique_id"].unique().tolist())

train_records = []
for _, row in train_balanced_df.iterrows():
    train_records.append({"text": make_training_text(row, tokenizer, allowed_labels)})

train_dataset = Dataset.from_list(train_records)

print(f"Training labels ({len(allowed_labels)}): {allowed_labels}")
print(f"Training samples: {len(train_dataset)}")
print(f"\nSample training text (first 500 chars):")
print(train_dataset[0]["text"][:500])

In [ ]:
# ============================================================================
# TRAINING: Enhanced Configuration
# ============================================================================

from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        gradient_checkpointing=True,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=1e-4,
        warmup_ratio=0.03,
        max_grad_norm=1.0,  # Gradient clipping
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=25,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir=str(OUTPUT_DIR / "qwen_lora_balanced_outputs"),
        report_to="none",
    ),
)

print("\n" + "="*60)
print(f"Starting training: {NUM_EPOCHS} epochs, {len(allowed_labels)} classes")
print("="*60)

trainer.train()

print("\n✓ Training complete")

In [ ]:
# ============================================================================
# INFERENCE & PREDICTION
# ============================================================================

FastLanguageModel.for_inference(model)

def extract_tcode(text):
    candidates = re.findall(r"T\d{4}(?:\.\d{3})?", text)
    for c in candidates:
        if c in allowed_labels:
            return c
    return candidates[0] if candidates else ""

def predict_one(alert_text):
    messages = [
        {"role": "system", "content": "You are a cybersecurity analyst mapping alerts to MITRE ATT&CK techniques."},
        {"role": "user", "content": make_prompt(alert_text, allowed_labels)}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([prompt], return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=12,
            do_sample=False,
            temperature=None,
            pad_token_id=tokenizer.eos_token_id,
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    generated = decoded[len(prompt):] if decoded.startswith(prompt) else decoded
    return extract_tcode(generated), generated.strip()

print("✓ Inference functions ready")

In [ ]:
# ============================================================================
# RUN PREDICTIONS ON TEST SET
# ============================================================================

pred_rows = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Predicting"):
    pred, raw = predict_one(row["text"])
    pred_rows.append({
        "alert_id": row["alert_id"],
        "gold_technique_id": row["gold_technique_id"],
        "predicted_technique_id": pred,
        "raw_model_output": raw,
        "alert_signature": row["alert_signature"],
        "text": row["text"],
    })

pred_df = pd.DataFrame(pred_rows)

print("\n" + "="*60)
print("PREDICTION DISTRIBUTION")
print("="*60)
print(pred_df["predicted_technique_id"].value_counts())

print("\n" + "="*60)
print("GOLD LABEL DISTRIBUTION (Test Set)")
print("="*60)
print(pred_df["gold_technique_id"].value_counts())

In [ ]:
# ============================================================================
# EVALUATION & DIAGNOSTICS
# ============================================================================

y_true = pred_df["gold_technique_id"].tolist()
y_pred = pred_df["predicted_technique_id"].tolist()

acc = accuracy_score(y_true, y_pred)
macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)

metrics = {
    "num_predictions": len(pred_df),
    "num_gold_classes": int(pd.Series(y_true).nunique()),
    "num_predicted_classes": int(pd.Series(y_pred).nunique()),
    "top1_accuracy": float(acc),
    "macro_precision": float(macro_p),
    "macro_recall": float(macro_r),
    "macro_f1": float(macro_f1),
    "weighted_precision": float(weighted_p),
    "weighted_recall": float(weighted_r),
    "weighted_f1": float(weighted_f1),
}

print("\n" + "="*60)
print("OVERALL METRICS")
print("="*60)
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"{k}: {v:.4f}")
    else:
        print(f"{k}: {v}")

# Per-class accuracy
print("\n" + "="*60)
print("PER-CLASS ACCURACY")
print("="*60)
for label in sorted(allowed_labels):
    correct = sum(1 for i in range(len(y_true)) if y_true[i] == label and y_pred[i] == label)
    total = sum(1 for i in range(len(y_true)) if y_true[i] == label)
    if total > 0:
        print(f"{label}: {correct}/{total} = {100*correct/total:.1f}%")

# Warning for label collapse
print("\n" + "="*60)
print("COLLAPSE DETECTION")
print("="*60)
top_pred = pred_df["predicted_technique_id"].value_counts().iloc[0] if len(pred_df) > 0 else 0
collapse_pct = 100 * top_pred / len(pred_df)
print(f"Top predicted label accounts for: {collapse_pct:.1f}% of predictions")
if collapse_pct > 70:
    print("⚠ WARNING: Possible label collapse detected!")
    print(f"  Top label: {pred_df['predicted_technique_id'].value_counts().index[0]}")
else:
    print("✓ Prediction distribution looks healthy")

In [ ]:
# ============================================================================
# CONFUSION MATRIX & CLASSIFICATION REPORT
# ============================================================================

# Classification report
report_dict = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
report_df = pd.DataFrame(report_dict).transpose()

print("\n" + "="*60)
print("CLASSIFICATION REPORT (Top Classes)")
print("="*60)
print(report_df.loc[allowed_labels])

# Confusion matrix
labels = sorted(set(y_true) | set(y_pred))
cm = confusion_matrix(y_true, y_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=labels, columns=labels)

print("\n" + "="*60)
print("CONFUSION MATRIX")
print("="*60)
print(cm_df)

In [ ]:
# ============================================================================
# SAVE RESULTS
# ============================================================================

metrics_path = OUTPUT_DIR / "metrics_llm_balanced.json"
pred_path = OUTPUT_DIR / "predictions_llm_balanced.jsonl"
report_path = OUTPUT_DIR / "classification_report_llm_balanced.csv"
conf_path = OUTPUT_DIR / "confusion_matrix_llm_balanced.csv"

with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

pred_df.to_json(pred_path, orient="records", lines=True)
report_df.to_csv(report_path)
cm_df.to_csv(conf_path)

print("Saved:")
print(f"  {metrics_path}")
print(f"  {pred_path}")
print(f"  {report_path}")
print(f"  {conf_path}")

In [ ]:
# Optional: Create results archive
try:
    import shutil
    archive = BASE / "llm_balanced_results"
    shutil.make_archive(str(archive), "zip", OUTPUT_DIR)
    print(f"✓ Results archived: {archive}.zip")
except Exception as e:
    print(f"Could not create archive: {e}")